In [5]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from AttributionPipeline.src.attribution_methods import AttributionMethod

# Sample SMILES strings for testing
SAMPLE_SMILES = [
    "Cc1ccc(C)cc1", # label 1
    "CC(=O)OC1=CC=CC=C1C(=O)O",  # Aspirin
    "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",  # Caffeine
    "CC(C)Cc1ccc(cc1)C(C)C(=O)O",  # Ibuprofen
]

def test_captum_methods():
    """Test all Captum-based attribution methods."""
    
    print("=" * 80)
    print("TESTING CAPTUM ATTRIBUTION METHODS")
    print("=" * 80)
    
    task_index = 0  # Change this to your desired task
    methods = ["integrated_gradients", "input_x_gradient"]
    
    for method_name in methods:
        print(f"\n{'='*80}")
        print(f"Method: {method_name.upper()}")
        print(f"{'='*80}")
        
        try:
            # Initialize attribution method
            attr_method = AttributionMethod(
                method_name=method_name,
                task_index=task_index,
                label_index=0,
                device="mps"
            )
            
            # Test on first SMILES
            smiles = SAMPLE_SMILES[0]
            print(f"\nTesting SMILES: {smiles}")
            
            # Get prediction
            pred_result = attr_method.predict(smiles)
            print(f"Prediction: {pred_result['pred_label']}")
            print(f"Prediction value: {pred_result['pred_value']:.4f}")
            
            # Compute attributions
            attributions, delta = attr_method.compute(
                input_ids=pred_result['input_ids'],
                attention_mask=pred_result['attention_mask'],
                n_steps=50,
                normalize=False,
                return_convergence_delta=True
            )
            
            print(f"\nAttribution shape: {attributions.shape}")
            if delta is not None:
                print(f"Convergence delta: {delta.item():.6f}")
            
            # Get top contributing tokens
            top_tokens = attr_method.get_top_tokens(
                tokens=pred_result['tokens'],
                attributions=attributions[0].numpy(),
                top_k=5,
                include_special=True
            )
            
            print("\nTop 5 contributing tokens:")
            for token, score in top_tokens:
                print(f"  {token:15s}: {score:+.6f}")
            
        except Exception as e:
            print(f"ERROR testing {method_name}: {str(e)}")
            import traceback
            traceback.print_exc()

def test_shap_method():
    """Test SHAP attribution method."""
    
    print("\n" + "=" * 80)
    print("TESTING SHAP ATTRIBUTION METHOD")
    print("=" * 80)
    
    task_index = 0
    
    try:
        # Initialize SHAP method
        attr_method = AttributionMethod(
            method_name="shap",
            task_index=task_index,
            label_index=0,
            device="mps"
        )
        
        # Test on first SMILES
        smiles = SAMPLE_SMILES[0]
        print(f"\nTesting SMILES: {smiles}")
        
        # Get prediction
        pred_result = attr_method.predict(smiles)
        print(f"Prediction: {pred_result['pred_label']}")
        print(f"Prediction value: {pred_result['pred_value']:.4f}")
        
        # Compute SHAP attributions
        print("\nComputing SHAP values (this may take a while)...")
        attributions, _ = attr_method.compute(
            input_ids=pred_result['input_ids'],
            attention_mask=pred_result['attention_mask']
        )
        
        print(f"\nAttribution shape: {attributions.shape}")
        
        # Get top contributing tokens
        top_tokens = attr_method.get_top_tokens(
            tokens=pred_result['tokens'],
            attributions=attributions[0].cpu().numpy(),
            top_k=5,
            include_special=True
        )
        
        print("\nTop 5 contributing tokens:")
        for token, score in top_tokens:
            print(f"  {token:15s}: {score:+.6f}")
        
        # Visualize with SHAP's built-in visualization
        shap_values = attr_method.visualize_shap(smiles)
        print("shap values:")
        print(shap_values)
        
    except Exception as e:
        print(f"ERROR testing SHAP: {str(e)}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    # Set random seeds for reproducibility
    torch.manual_seed(42)
    np.random.seed(42)
    
    # Run tests
    print("Starting attribution method tests...\n")
    
    # Test 1: Individual Captum methods
    test_captum_methods()
    
    # Test 2: SHAP method (comment out if too slow)
    test_shap_method()
    
    print("\n" + "=" * 80)
    print("ALL TESTS COMPLETED")
    print("=" * 80)
    

Starting attribution method tests...

TESTING CAPTUM ATTRIBUTION METHODS

Method: INTEGRATED_GRADIENTS
Loading model from Weights/Scaffold_CheMLT-F...
768 768


Some weights of DebertaV2Model were not initialized from the model checkpoint at Weights/Scaffold_CheMLT-F and are newly initialized: ['embeddings.LayerNorm.bias', 'embeddings.LayerNorm.weight', 'embeddings.position_embeddings.weight', 'embeddings.word_embeddings.weight', 'encoder.layer.0.attention.output.LayerNorm.bias', 'encoder.layer.0.attention.output.LayerNorm.weight', 'encoder.layer.0.attention.output.dense.bias', 'encoder.layer.0.attention.output.dense.weight', 'encoder.layer.0.attention.self.key_proj.bias', 'encoder.layer.0.attention.self.key_proj.weight', 'encoder.layer.0.attention.self.query_proj.bias', 'encoder.layer.0.attention.self.query_proj.weight', 'encoder.layer.0.attention.self.value_proj.bias', 'encoder.layer.0.attention.self.value_proj.weight', 'encoder.layer.0.intermediate.dense.bias', 'encoder.layer.0.intermediate.dense.weight', 'encoder.layer.0.output.LayerNorm.bias', 'encoder.layer.0.output.LayerNorm.weight', 'encoder.layer.0.output.dense.bias', 'encoder.layer.0

✅ Loaded from model.safetensors
✅ Tokenizer loaded


Testing SMILES: Cc1ccc(C)cc1
Prediction: Active (prob=0.532)
Prediction value: 0.5318

Attribution shape: torch.Size([1, 512])
Convergence delta: -0.022024

Top 5 contributing tokens:
  ccc            : -0.114389
  Cc             : -0.073499
  (              : -0.032700
  1              : -0.018380
  1              : +0.009198

Method: INPUT_X_GRADIENT
Loading model from Weights/Scaffold_CheMLT-F...
768 768


Some weights of DebertaV2Model were not initialized from the model checkpoint at Weights/Scaffold_CheMLT-F and are newly initialized: ['embeddings.LayerNorm.bias', 'embeddings.LayerNorm.weight', 'embeddings.position_embeddings.weight', 'embeddings.word_embeddings.weight', 'encoder.layer.0.attention.output.LayerNorm.bias', 'encoder.layer.0.attention.output.LayerNorm.weight', 'encoder.layer.0.attention.output.dense.bias', 'encoder.layer.0.attention.output.dense.weight', 'encoder.layer.0.attention.self.key_proj.bias', 'encoder.layer.0.attention.self.key_proj.weight', 'encoder.layer.0.attention.self.query_proj.bias', 'encoder.layer.0.attention.self.query_proj.weight', 'encoder.layer.0.attention.self.value_proj.bias', 'encoder.layer.0.attention.self.value_proj.weight', 'encoder.layer.0.intermediate.dense.bias', 'encoder.layer.0.intermediate.dense.weight', 'encoder.layer.0.output.LayerNorm.bias', 'encoder.layer.0.output.LayerNorm.weight', 'encoder.layer.0.output.dense.bias', 'encoder.layer.0

✅ Loaded from model.safetensors
✅ Tokenizer loaded


Testing SMILES: Cc1ccc(C)cc1
Prediction: Active (prob=0.532)
Prediction value: 0.5318

Attribution shape: torch.Size([1, 512])

Top 5 contributing tokens:
  [SEP]          : -0.024892
  cc             : +0.010649
  1              : +0.007788
  (              : +0.006456
  C              : +0.005320

TESTING SHAP ATTRIBUTION METHOD
Loading model from Weights/Scaffold_CheMLT-F...
768 768


Some weights of DebertaV2Model were not initialized from the model checkpoint at Weights/Scaffold_CheMLT-F and are newly initialized: ['embeddings.LayerNorm.bias', 'embeddings.LayerNorm.weight', 'embeddings.position_embeddings.weight', 'embeddings.word_embeddings.weight', 'encoder.layer.0.attention.output.LayerNorm.bias', 'encoder.layer.0.attention.output.LayerNorm.weight', 'encoder.layer.0.attention.output.dense.bias', 'encoder.layer.0.attention.output.dense.weight', 'encoder.layer.0.attention.self.key_proj.bias', 'encoder.layer.0.attention.self.key_proj.weight', 'encoder.layer.0.attention.self.query_proj.bias', 'encoder.layer.0.attention.self.query_proj.weight', 'encoder.layer.0.attention.self.value_proj.bias', 'encoder.layer.0.attention.self.value_proj.weight', 'encoder.layer.0.intermediate.dense.bias', 'encoder.layer.0.intermediate.dense.weight', 'encoder.layer.0.output.LayerNorm.bias', 'encoder.layer.0.output.LayerNorm.weight', 'encoder.layer.0.output.dense.bias', 'encoder.layer.0

✅ Loaded from model.safetensors
✅ Tokenizer loaded


Testing SMILES: Cc1ccc(C)cc1
Prediction: Active (prob=0.532)
Prediction value: 0.5318

Computing SHAP values (this may take a while)...


PartitionExplainer explainer: 2it [00:18, 18.74s/it]               



Attribution shape: torch.Size([1, 512])

Top 5 contributing tokens:
  [SEP]          : -0.027003
  cc             : -0.013260
  1              : +0.009722
  Cc             : -0.009277
  1              : -0.008661
Prediction: [0.46817201 0.53182799]


shap values:
.values =
array([[[ 0.        ,  0.        ],
        [-0.03523492,  0.03523492],
        [-0.00098795,  0.00098795],
        [-0.03551683,  0.03551683],
        [-0.01047166,  0.01047166],
        [-0.23607729,  0.23607729],
        [-0.00535239,  0.00535239],
        [-0.10041772,  0.10041772],
        [ 0.02728695, -0.02728695],
        [ 0.        ,  0.        ]]])

.base_values =
array([[0.8649438, 0.1350562]])

.data =
(array(['', 'Cc', '1', 'ccc', '(', 'C', ')', 'cc', '1', ''], dtype=object),)

ALL TESTS COMPLETED
